# Reimplementing the first example from scratch

A Dense layer, a Sequential model, a batch generator, and a training step — written by hand, then checked against Keras. Nothing here is how you should work; everything here is what fit() is doing.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 2 — The Mathematical Building Blocks of Neural Networks](../../../course-web-slides/ch02/index.html) &nbsp;·&nbsp; **Section:** 04 — Looking back at our first example

---

## A Dense layer, by hand

In [ ]:
import tensorflow as tf
import numpy as np

class NaiveDense:
    def __init__(self, input_size, output_size, activation):
        self.activation = activation
        w_shape = (input_size, output_size)
        w_initial = tf.random.uniform(w_shape, minval=0, maxval=1e-1)
        self.W = tf.Variable(w_initial)
        self.b = tf.Variable(tf.zeros((output_size,)))

    def __call__(self, inputs):
        return self.activation(tf.matmul(inputs, self.W) + self.b)

    @property
    def weights(self):
        return [self.W, self.b]

Three lines of substance: a matrix multiply, an addition, an activation. Everything else is bookkeeping.

## A Sequential model, by hand

In [ ]:
class NaiveSequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer(x)
        return x

    @property
    def weights(self):
        return [w for layer in self.layers for w in layer.weights]

model = NaiveSequential([
    NaiveDense(28 * 28, 512, activation=tf.nn.relu),
    NaiveDense(512, 10, activation=tf.nn.softmax),
])
assert len(model.weights) == 4
print(f"{sum(int(np.prod(w.shape)) for w in model.weights):,} parameters")

Expected output:

```
407,050 parameters
```

## A batch generator

In [ ]:
class BatchGenerator:
    def __init__(self, images, labels, batch_size=128):
        assert len(images) == len(labels)
        self.index = 0
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        self.num_batches = math.ceil(len(images) / batch_size)

    def next(self):
        images = self.images[self.index:self.index + self.batch_size]
        labels = self.labels[self.index:self.index + self.batch_size]
        self.index += self.batch_size
        return images, labels

import math

## One training step

Forward pass under a tape, loss, gradients, update. **This is the whole of `fit()`**, minus callbacks, metrics, and every convenience.

In [ ]:
learning_rate = 1e-3

def update_weights(gradients, weights):
    for g, w in zip(gradients, weights):
        w.assign_sub(g * learning_rate)

def one_training_step(model, images_batch, labels_batch):
    with tf.GradientTape() as tape:
        predictions = model(images_batch)
        per_sample_losses = tf.keras.losses.sparse_categorical_crossentropy(
            labels_batch, predictions
        )
        average_loss = tf.reduce_mean(per_sample_losses)
    gradients = tape.gradient(average_loss, model.weights)
    update_weights(gradients, model.weights)
    return average_loss

> **Note** — `w.assign_sub(g * lr)` is the entire optimizer. Replace this one line and you have SGD with momentum, or RMSprop, or Adam — that is genuinely the only difference between them.

## The training loop

In [ ]:
def fit(model, images, labels, epochs, batch_size=128):
    for epoch_counter in range(epochs):
        print(f"Epoch {epoch_counter}")
        batch_generator = BatchGenerator(images, labels, batch_size)
        for batch_counter in range(batch_generator.num_batches):
            images_batch, labels_batch = batch_generator.next()
            loss = one_training_step(model, images_batch, labels_batch)
            if batch_counter % 100 == 0:
                print(f"  loss at batch {batch_counter}: {loss:.2f}")

from keras.datasets import mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape((60000, 28 * 28)).astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255

fit(model, train_images, train_labels, epochs=10, batch_size=128)

Expected output:

```
Epoch 0
  loss at batch 0: 5.02
  loss at batch 100: 2.23
  ...
Epoch 9
  loss at batch 400: 0.19
```

## Did it work?

In [ ]:
predictions = model(test_images).numpy()
predicted_labels = np.argmax(predictions, axis=1)
matches = predicted_labels == test_labels
print(f"accuracy: {matches.mean():.3f}")

Expected output:

```
accuracy: 0.81 — 0.83
```

Around 82%, against 97% from the Keras version in notebook 01. **Same architecture, same data.** The difference is the optimizer — plain SGD at a fixed 1e-3, against RMSprop — and it is a fifteen-point difference.

That is a useful number to carry: the parts of Keras that look like convenience are frequently not.

## Checking one gradient against Keras

In [ ]:
import keras
from keras import layers

kmodel = keras.Sequential([layers.Dense(4, activation="relu"),
                           layers.Dense(3, activation="softmax")])
xb = tf.random.normal((8, 5))
yb = tf.constant([0, 1, 2, 0, 1, 2, 0, 1])

with tf.GradientTape() as tape:
    loss = tf.reduce_mean(
        keras.losses.sparse_categorical_crossentropy(yb, kmodel(xb))
    )
g = tape.gradient(loss, kmodel.trainable_weights)
print("gradient shapes:", [tuple(t.shape) for t in g])
print("finite:", all(bool(tf.reduce_all(tf.math.is_finite(t))) for t in g))

Same mechanism, same tape. Keras is not doing anything else — it is doing this, with the loop, the metrics, and the callbacks written for you.

---

## What to take away

- `fit()` is a batch generator, a gradient tape, and one `assign_sub` per weight.
- The optimizer is one line, and swapping it is worth fifteen points of accuracy here.
- Writing it by hand once is what makes the framework legible afterwards.
- You should never work this way — but you should know exactly what you are calling.